# Lesson 4: Cleaning the Books - Data Quality & Null Handling

---

## The Story

The data is messy -- like a property with maintenance issues.
Before you can value or sell it, you need a thorough inspection and cleanup.

You are the **data inspector** today. Your job: find every issue in the dataset,
document it, and apply the appropriate fix -- just like a property inspector
identifies defects and a contractor repairs them.

---

## What You Will Learn

- How to **detect nulls** and measure data quality per column
- **`dropna()`** -- remove rows with null values
- **`fillna()`** -- fill null values with sensible defaults
- **`when()` / `otherwise()`** -- conditional logic for smart null handling
- **Type casting** -- convert columns to the correct data types
- Building a **full cleaning pipeline** that chains all steps together


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, isnan, to_date
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, FloatType, DateType
)

spark = SparkSession.builder \
    .appName("CleaningTheBooks") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Load with inferSchema so we can see what raw types look like
df_raw = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/property_listings.csv")

print(f"Loaded {df_raw.count()} rows with {len(df_raw.columns)} columns")
df_raw.show(3)

Loaded 510 rows with 14 columns


+-----------+-------------+--------+--------+-------------+--------+---------+------+----------+----------+------------+-------+------------+--------+
|property_id|      address|    city|zip_code|property_type|bedrooms|bathrooms|  sqft|list_price|year_built|neighborhood| status|listing_date|agent_id|
+-----------+-------------+--------+--------+-------------+--------+---------+------+----------+----------+------------+-------+------------+--------+
|  PROP-0001|7370 River Rd|Parkview|   84038|        House|       5|      1.0|  NULL| 1268597.0|    1988.0|   Riverside|   Sold|  2025-12-24| AGT-016|
|  PROP-0002|960 Park Blvd|Parkview|   62393|        House|    NULL|      2.0|4859.0|  835503.0|    1955.0|     Oakwood|   NULL|  2025-10-07| AGT-004|
|  PROP-0003| 5291 Hill Dr|Parkview|   76081|        House|       3|      1.0|4882.0|  487283.0|    2018.0|    Hillside|Pending|        NULL| AGT-017|
+-----------+-------------+--------+--------+-------------+--------+---------+------+---------

## The Property Inspection - Assessing Data Quality

Before fixing anything, a good inspector **documents every defect**.
The equivalent in data is counting nulls, checking data types,
and understanding what percentage of each column is missing.

### The Null Audit Pattern

```python
from pyspark.sql.functions import col, when, count

# Count nulls in each column using a list comprehension
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()
```

> **Property analogy:** This is your inspection checklist -- every column is a system
> (plumbing, electrical, roof), and you are counting how many items need attention.


In [2]:
# Count nulls per column using a list comprehension
print("=== NULL COUNT PER COLUMN ===")
null_counts = df_raw.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
])
null_counts.show()

total_rows = df_raw.count()
print(f"Total rows in dataset: {total_rows}")

=== NULL COUNT PER COLUMN ===


+-----------+-------+----+--------+-------------+--------+---------+----+----------+----------+------------+------+------------+--------+
|property_id|address|city|zip_code|property_type|bedrooms|bathrooms|sqft|list_price|year_built|neighborhood|status|listing_date|agent_id|
+-----------+-------+----+--------+-------------+--------+---------+----+----------+----------+------------+------+------------+--------+
|          0|      0|   0|       0|           31|      20|       39|  51|        29|        90|          23|    26|          66|      50|
+-----------+-------+----+--------+-------------+--------+---------+----+----------+----------+------------+------+------------+--------+

Total rows in dataset: 510


In [3]:
# Calculate null percentage per column and print a formatted report
total_rows = df_raw.count()

print("=" * 52)
print("   DATA QUALITY REPORT - NULL PERCENTAGE")
print("=" * 52)
print(f"  {'Column':<20} {'Null Count':>10} {'Null %':>8}")
print("-" * 52)

null_vals = null_counts.collect()[0].asDict()
for col_name in df_raw.columns:
    n = null_vals[col_name]
    pct = (n / total_rows) * 100
    flag = "  <-- NEEDS ATTENTION" if pct > 5 else ""
    print(f"  {col_name:<20} {n:>10} {pct:>7.1f}%{flag}")

print("=" * 52)

   DATA QUALITY REPORT - NULL PERCENTAGE
  Column               Null Count   Null %
----------------------------------------------------


  property_id                   0     0.0%
  address                       0     0.0%
  city                          0     0.0%
  zip_code                      0     0.0%
  property_type                31     6.1%  <-- NEEDS ATTENTION
  bedrooms                     20     3.9%
  bathrooms                    39     7.6%  <-- NEEDS ATTENTION
  sqft                         51    10.0%  <-- NEEDS ATTENTION
  list_price                   29     5.7%  <-- NEEDS ATTENTION
  year_built                   90    17.6%  <-- NEEDS ATTENTION
  neighborhood                 23     4.5%
  status                       26     5.1%  <-- NEEDS ATTENTION
  listing_date                 66    12.9%  <-- NEEDS ATTENTION
  agent_id                     50     9.8%  <-- NEEDS ATTENTION


## Removing Nulls - dropna()

`dropna()` removes rows that contain null values. You can control how aggressively:

| Method | Behavior |
|--------|----------|
| `df.dropna()` | Drop row if **any** column is null (default: `how='any'`) |
| `df.dropna(how='all')` | Drop row only if **all** columns are null |
| `df.dropna(subset=['a','b'])` | Drop row if null in specified columns only |
| `df.dropna(thresh=5)` | Drop row if fewer than 5 non-null values |

> **Property analogy:** `dropna(how='any')` is like rejecting any property with
> a single defect. `dropna(subset=[...])` is like only rejecting properties
> with issues in the critical systems (foundation, roof).


In [4]:
total = df_raw.count()
print(f"Original row count: {total}")
print()

# Drop rows where ANY column is null
df_drop_any = df_raw.dropna(how="any")
dropped_any = total - df_drop_any.count()
print(f"After dropna(how='any'):                 {df_drop_any.count()} rows  ({dropped_any} dropped)")

# Drop rows only where ALL columns are null
df_drop_all = df_raw.dropna(how="all")
dropped_all = total - df_drop_all.count()
print(f"After dropna(how='all'):                 {df_drop_all.count()} rows  ({dropped_all} dropped)")

# Drop rows only missing critical business columns
df_drop_critical = df_raw.dropna(subset=["list_price", "sqft"])
dropped_critical = total - df_drop_critical.count()
print(f"After dropna(subset=[list_price, sqft]): {df_drop_critical.count()} rows  ({dropped_critical} dropped)")

print("\n[Recommendation] Use subset= to target only business-critical columns.")

Original row count: 510



After dropna(how='any'):                 207 rows  (303 dropped)


After dropna(how='all'):                 510 rows  (0 dropped)


After dropna(subset=[list_price, sqft]): 434 rows  (76 dropped)

[Recommendation] Use subset= to target only business-critical columns.


## Filling Nulls - fillna()

Instead of dropping rows, `fillna()` replaces nulls with a default value.
This preserves your data while making it processable.

```python
# Fill all nulls with a single value
df.fillna(0)

# Fill specific columns with different values (preferred)
df.fillna({
    "numeric_col": 0,
    "string_col":  "Unknown",
})
```

> **Property analogy:** `fillna` is like marking unknown values as "TBD" on your
> inspection form instead of leaving blanks -- the form stays complete and processable.


In [5]:
# Check null counts before fillna
print("=== BEFORE fillna ===")
df_raw.select(
    count(when(col("property_type").isNull(), 1)).alias("property_type_nulls"),
    count(when(col("bedrooms").isNull(), 1)).alias("bedrooms_nulls"),
    count(when(col("neighborhood").isNull(), 1)).alias("neighborhood_nulls")
).show()

# Fill nulls with appropriate defaults per column
df_filled = df_raw.fillna({
    "property_type": "Unknown",
    "bedrooms":      0,
    "bathrooms":     0.0,
    "neighborhood":  "Unspecified",
    "status":        "Unknown"
})

# Verify nulls are gone for the filled columns
print("=== AFTER fillna ===")
df_filled.select(
    count(when(col("property_type").isNull(), 1)).alias("property_type_nulls"),
    count(when(col("bedrooms").isNull(), 1)).alias("bedrooms_nulls"),
    count(when(col("neighborhood").isNull(), 1)).alias("neighborhood_nulls")
).show()

print("Sample rows where neighborhood was filled as Unspecified:")
df_filled.filter(col("neighborhood") == "Unspecified") \
    .select("address", "city", "neighborhood").show(5)

=== BEFORE fillna ===


+-------------------+--------------+------------------+
|property_type_nulls|bedrooms_nulls|neighborhood_nulls|
+-------------------+--------------+------------------+
|                 31|            20|                23|
+-------------------+--------------+------------------+

=== AFTER fillna ===


+-------------------+--------------+------------------+
|property_type_nulls|bedrooms_nulls|neighborhood_nulls|
+-------------------+--------------+------------------+
|                  0|             0|                 0|
+-------------------+--------------+------------------+

Sample rows where neighborhood was filled as Unspecified:


+--------------+-----------+------------+
|       address|       city|neighborhood|
+--------------+-----------+------------+
| 9770 River Rd|  Riverside| Unspecified|
| 4959 River Rd|Springfield| Unspecified|
|  8671 Hill Dr|    Oakwood| Unspecified|
|  5692 Main St|    Oakwood| Unspecified|
|7626 Park Blvd|Springfield| Unspecified|
+--------------+-----------+------------+
only showing top 5 rows


## Conditional Logic with when/otherwise

`when()` and `otherwise()` give you the power of SQL CASE WHEN expressions.
They are perfect for context-aware null filling and creating categorical labels.

```python
from pyspark.sql.functions import when

df.withColumn(
    "price_tier",
    when(col("list_price") < 300000,  "Budget")
    .when(col("list_price") < 600000, "Mid-Range")
    .when(col("list_price") < 1000000,"Premium")
    .otherwise("Luxury")
)
```

> **Property analogy:** `when/otherwise` is like the appraiser's rubric:
> 'If condition A, grade X; if condition B, grade Y; otherwise grade Z.'


In [6]:
# Use when/otherwise to intelligently fill missing neighborhoods based on city
df_smart = df_raw.withColumn(
    "neighborhood",
    when(col("neighborhood").isNotNull(), col("neighborhood"))
    .when(col("city") == "Austin",   "Austin-General")
    .when(col("city") == "Dallas",   "Dallas-General")
    .when(col("city") == "Houston",  "Houston-General")
    .otherwise("Unspecified")
)

# Categorize bedrooms into human-readable size labels
df_smart = df_smart.withColumn(
    "size_category",
    when(col("bedrooms").isNull(), "Unknown")
    .when(col("bedrooms") <= 1,    "Studio/1BR")
    .when(col("bedrooms") <= 2,    "Small")
    .when(col("bedrooms") <= 3,    "Medium")
    .when(col("bedrooms") <= 5,    "Large")
    .otherwise("Estate")
)

print("=== Size Category Distribution ===")
df_smart.groupBy("size_category").count().orderBy("size_category").show()

print("=== Sample rows with conditional fills ===")
df_smart.select(
    "address", "city", "neighborhood", "bedrooms", "size_category"
).show(8)

=== Size Category Distribution ===


+-------------+-----+
|size_category|count|
+-------------+-----+
|        Large|  182|
|       Medium|  160|
|        Small|   95|
|   Studio/1BR|   53|
|      Unknown|   20|
+-------------+-----+

=== Sample rows with conditional fills ===
+-------------+-----------+------------+--------+-------------+
|      address|       city|neighborhood|bedrooms|size_category|
+-------------+-----------+------------+--------+-------------+
|7370 River Rd|   Parkview|   Riverside|       5|        Large|
|960 Park Blvd|   Parkview|     Oakwood|    NULL|      Unknown|
| 5291 Hill Dr|   Parkview|    Hillside|       3|       Medium|
| 5834 Oak Ave|Springfield|    Downtown|       3|       Medium|
|566 Park Blvd|   Parkview|    Downtown|       4|        Large|
| 5678 Hill Dr|Springfield|     Seaside|       5|        Large|
| 8422 Hill Dr|   Parkview|  Greenfield|       3|       Medium|
| 869 River Rd|    Oakwood|    Parkview|       3|       Medium|
+-------------+-----------+------------+--------+-----

## Type Casting

Even with `inferSchema=True`, Spark sometimes gets types wrong.
Explicit casting ensures columns have exactly the right types
for calculations, sorting, and joining.

```python
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.functions import to_date

# Cast numeric columns
df.withColumn("bedrooms", col("bedrooms").cast(IntegerType()))

# Cast date strings to DateType
df.withColumn("listing_date", to_date(col("listing_date"), "yyyy-MM-dd"))
```

> **Caution:** If a value cannot be cast (e.g., casting `"abc"` to Integer),
> Spark will silently return `null` rather than raising an error.
> Always check for new nulls after casting.


In [7]:
# Cast columns to proper types
df_typed = df_raw \
    .withColumn("bedrooms",     col("bedrooms").cast(IntegerType())) \
    .withColumn("bathrooms",    col("bathrooms").cast(FloatType())) \
    .withColumn("sqft",         col("sqft").cast(IntegerType())) \
    .withColumn("list_price",   col("list_price").cast(DoubleType())) \
    .withColumn("year_built",   col("year_built").cast(IntegerType())) \
    .withColumn("listing_date", to_date(col("listing_date"), "yyyy-MM-dd"))

print("=== Schema BEFORE type casting ===")
df_raw.printSchema()

print("=== Schema AFTER type casting ===")
df_typed.printSchema()

# Check if casting introduced any new nulls in key columns
print("=== Null check after casting ===")
df_typed.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["bedrooms", "sqft", "list_price", "year_built", "listing_date"]
]).show()

=== Schema BEFORE type casting ===
root
 |-- property_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- zip_code: integer (nullable = true)
 |-- property_type: string (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- sqft: double (nullable = true)
 |-- list_price: double (nullable = true)
 |-- year_built: double (nullable = true)
 |-- neighborhood: string (nullable = true)
 |-- status: string (nullable = true)
 |-- listing_date: date (nullable = true)
 |-- agent_id: string (nullable = true)

=== Schema AFTER type casting ===
root
 |-- property_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- zip_code: integer (nullable = true)
 |-- property_type: string (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: float (nullable = true)
 |-- sqft: integer (nullable = true)
 |-- list_price: double (nul

+--------+----+----------+----------+------------+
|bedrooms|sqft|list_price|year_built|listing_date|
+--------+----+----------+----------+------------+
|      20|  51|        29|        90|          66|
+--------+----+----------+----------+------------+



In [8]:
# Full cleanup pipeline -- chain all cleaning steps together

print("=" * 55)
print("   FULL DATA CLEANING PIPELINE")
print("=" * 55)

before_rows = df_raw.count()
before_nulls_row = df_raw.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_raw.columns
]).collect()[0].asDict()
print(f"\nBEFORE: {before_rows} rows, {sum(before_nulls_row.values())} total null cells")

# Step 1: Cast all columns to correct types
# Step 2: Fill nulls with sensible defaults
# Step 3: Smart neighborhood fill using city
# Step 4: Drop rows still missing critical numeric data
df_clean = df_raw \
    .withColumn("bedrooms",     col("bedrooms").cast(IntegerType())) \
    .withColumn("bathrooms",    col("bathrooms").cast(FloatType())) \
    .withColumn("sqft",         col("sqft").cast(IntegerType())) \
    .withColumn("list_price",   col("list_price").cast(DoubleType())) \
    .withColumn("year_built",   col("year_built").cast(IntegerType())) \
    .withColumn("listing_date", to_date(col("listing_date"), "yyyy-MM-dd")) \
    .fillna({
        "property_type": "Unknown",
        "bedrooms":      0,
        "bathrooms":     0.0,
        "status":        "Unknown"
    }) \
    .withColumn(
        "neighborhood",
        when(col("neighborhood").isNotNull(), col("neighborhood"))
        .when(col("city") == "Austin",  "Austin-General")
        .when(col("city") == "Dallas",  "Dallas-General")
        .when(col("city") == "Houston", "Houston-General")
        .otherwise("Unspecified")
    ) \
    .dropna(subset=["list_price", "sqft"])

after_rows = df_clean.count()
after_nulls_row = df_clean.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_clean.columns
]).collect()[0].asDict()
print(f"AFTER:  {after_rows} rows, {sum(after_nulls_row.values())} total null cells")
print(f"Rows removed: {before_rows - after_rows}")

print("\n=== Clean DataFrame Schema ===")
df_clean.printSchema()
print("=" * 55)

   FULL DATA CLEANING PIPELINE



BEFORE: 510 rows, 425 total null cells


AFTER:  434 rows, 176 total null cells
Rows removed: 76

=== Clean DataFrame Schema ===
root
 |-- property_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- zip_code: integer (nullable = true)
 |-- property_type: string (nullable = false)
 |-- bedrooms: integer (nullable = false)
 |-- bathrooms: float (nullable = false)
 |-- sqft: integer (nullable = true)
 |-- list_price: double (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- neighborhood: string (nullable = true)
 |-- status: string (nullable = false)
 |-- listing_date: date (nullable = true)
 |-- agent_id: string (nullable = true)



## Lesson 4 Wrap-Up

You have completed a full data quality inspection and cleanup of the property listings.
This is the foundation of any reliable analysis.

### Key Takeaways

| Task | Method | Use When |
|------|--------|----------|
| Detect nulls | `count(when(col.isNull(), c))` | Audit data quality |
| Drop null rows | `df.dropna()` | Data completeness is critical |
| Fill nulls | `df.fillna({col: value})` | Preserve rows, add defaults |
| Conditional fill | `when().otherwise()` | Context-aware null handling |
| Fix data types | `col.cast(Type())` | Correct types for calculations |
| Parse dates | `to_date(col, format)` | Convert strings to DateType |

### The Cleaning Pipeline Pattern

```python
df_clean = df_raw \
    .withColumn("col", col("col").cast(IntegerType()))  # 1. Cast types
    .fillna({"col": default_value})                     # 2. Fill nulls
    .withColumn("col", when(...).otherwise(...))         # 3. Conditional logic
    .dropna(subset=["critical_col"])                     # 4. Drop unfixable rows
```

### Data Quality Checklist

- [ ] Null counts per column
- [ ] Null percentage (flag columns > 5%)
- [ ] Correct data types (numbers as numbers, dates as dates)
- [ ] Consistent categorical values (no mixed casing or trailing spaces)
- [ ] Row count before and after cleaning

---

**Next: Lesson 5 - Valuing the Portfolio**

With clean data in hand, you are ready for real analytics --
aggregations, `groupBy`, and portfolio-level statistics.
